In [1]:
# Imports
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam


2025-10-05 14:27:13.430933: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Defining the model with hyperparams

In [2]:
# Hypermodel
def build_model(hp):
    model = Sequential([
        Flatten(input_shape=(28,28)),
        Dense(
            units=hp.Int('units', min_value=32,
            max_value=512, step=32), activation='softmax'
            )
    ])
    model.compile(
        optimizer=Adam(
            learning_rate=hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


In [3]:
#configuring the search
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='intro_to_kt'
)


I0000 00:00:1759654636.537221  287200 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2128 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5
/home/shobhit/.local/lib/python3.9/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


# Running the hyperparameters search

In [4]:
# Data + search
(x_train, y_train), (x_val, y_val) = mnist.load_data()
x_train = x_train / 255.0
x_val = x_val / 255.0

tuner.search(
    x_train, y_train,
    epochs=5,
    validation_data=(x_val, y_val),
    verbose=1
)

best_model = tuner.get_best_models(1)[0]
best_hps = tuner.get_best_hyperparameters(1)[0]
print("Best hidden_units:", best_hps.get('hidden_units'))
print("Best learning_rate:", best_hps.get('learning_rate'))

Trial 10 Complete [00h 00m 31s]
val_accuracy: 0.9262499809265137

Best val_accuracy So Far: 0.9263499975204468
Total elapsed time: 00h 05m 06s


/home/shobhit/.local/lib/python3.9/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


KeyError: 'hidden_units does not exist.'